# Phase 1 — Data Exploration

Interactive companion to `scripts/explore_dataset.py`. Run the setup cell first, then
pick whichever dataset section you want to dig into — each is independent.

- **NSL-KDD**: freely downloaded via `scripts/download_nsl_kdd.py`. Flow-level, used as the flow-based comparison baseline (Phase 6).
- **CICIDS2017**: requires manual download (see `data/raw/cicids2017/README.md`). This is the payload-rich dataset that actually drives Phases 2-5.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing.nsl_kdd_columns import NSL_KDD_COLUMNS, label_to_category

NSL_KDD_DIR = PROJECT_ROOT / "data" / "raw" / "nsl-kdd"
CICIDS_DIR = PROJECT_ROOT / "data" / "raw" / "cicids2017"

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

## Part A — NSL-KDD

In [ ]:
train = pd.read_csv(NSL_KDD_DIR / "KDDTrain+.txt", names=NSL_KDD_COLUMNS)
test = pd.read_csv(NSL_KDD_DIR / "KDDTest+.txt", names=NSL_KDD_COLUMNS)

train["category"] = train["label"].apply(label_to_category)
test["category"] = test["label"].apply(label_to_category)

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

In [ ]:
# Attack category distribution — train vs test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["category"].value_counts().plot(kind="bar", ax=axes[0], title="Train — Category Distribution")
test["category"].value_counts().plot(kind="bar", ax=axes[1], title="Test — Category Distribution", color="orange")
plt.tight_layout()
plt.show()

In [ ]:
# Class imbalance check — this matters a lot for how you split train/val for
# the semi-supervised autoencoder (Phase 5), which should train on normal-only data
print("Train category %:")
print((100 * train["category"].value_counts(normalize=True)).round(2))
print()
print(f"Note: NSL-KDD's test set contains attack subtypes NOT present in training \n"
      f"(that's intentional — it's meant to test generalization to novel attacks).")

In [ ]:
# Numeric feature overview — which features actually vary, which are near-constant
numeric_cols = train.select_dtypes(include=[np.number]).columns.drop("difficulty_level", errors="ignore")
train[numeric_cols].describe().T.sort_values("std", ascending=False).head(15)

In [ ]:
# protocol_type / service breakdown — relevant later for per-service profiling (Phase 4)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train["protocol_type"].value_counts().plot(kind="bar", ax=axes[0], title="protocol_type")
train["service"].value_counts().head(15).plot(kind="bar", ax=axes[1], title="Top 15 services")
plt.tight_layout()
plt.show()

## Part B — CICIDS2017

Requires manual download first — see `data/raw/cicids2017/README.md`. Run
`python scripts/verify_cicids2017.py` beforehand to confirm files are in place.

In [ ]:
csv_files = sorted(CICIDS_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s)")
for f in csv_files:
    print(" -", f.name)

In [ ]:
# Load one day at a time -- loading all days into memory at once is unnecessary
# for exploration and can be a few GB combined.
# Start with Monday (benign-only baseline) since that's what the PAYL profile
# in Phase 4 will be built from.
monday_candidates = [f for f in csv_files if "monday" in f.name.lower()]
if monday_candidates:
    monday = pd.read_csv(monday_candidates[0], low_memory=False)
    monday.columns = [c.strip() for c in monday.columns]
    print(monday.shape)
    monday["Label"].value_counts()
else:
    print("Monday CSV not found yet -- see data/raw/cicids2017/README.md")

In [ ]:
# Combined label distribution across all downloaded days
all_counts = {}
total_rows = 0
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    if "Label" in df.columns:
        for label, count in df["Label"].value_counts().items():
            all_counts[label] = all_counts.get(label, 0) + count
        total_rows += len(df)

combined = pd.Series(all_counts).sort_values(ascending=False)
print(f"Total rows: {total_rows:,}")
combined

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
combined.plot(kind="barh", ax=ax, color="#55A868")
ax.set_title("CICIDS2017 — Label Distribution (All Downloaded Days)")
ax.set_xlabel("Record count")
plt.tight_layout()
plt.show()

In [ ]:
# Data quality check -- CICIDS2017 is known to have some inf/NaN values in a
# handful of flow-rate columns (division by zero for very short flows).
# Worth knowing about now so Phase 3 feature extraction handles it deliberately
# rather than silently propagating NaNs into a model.
if monday_candidates:
    numeric = monday.select_dtypes(include=[np.number])
    print("NaN count:", monday.isnull().sum().sum())
    print("Inf count:", np.isinf(numeric).sum().sum())

## Notes / takeaways to carry into Phase 2

- NSL-KDD: normal/attack roughly balanced (53%/47%) but attack sub-categories are wildly imbalanced (R2L 0.8%, U2R 0.04%) -- something to keep in mind if used for any comparison classifier in Phase 6.
- CICIDS2017: Monday is the clean "normal-only" day -- earmark it for building the PAYL per-service normal profile in Phase 4.
- CICIDS2017 CSVs can contain inf/NaN in rate-based columns -- Phase 3 feature extraction needs to handle this explicitly.
- Payload bytes only come from the PCAPs, not the CSVs -- the CSVs are flow-level summaries. Phase 2 parsing works against `data/raw/cicids2017/PCAPs/`.